# Session 17: Forests, boosting and overfitting

Prepare the Nile Microfinance loans, fit a logistic regression, a forest and a boosted model, tune the forest, and decide whether the extra complexity paid. Scores in your run will be close to the typical ones in the lesson, so the checks accept a range.

**Student version.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Prepare the loans
Read `loans.csv` into `loans`. Replace ages outside 18 to 90 with the median age. Fill blank incomes with the median income. Add `burden`, the monthly instalment divided by monthly income: `loan_amount_ugx` divided by `term_months`, divided by `monthly_income_ugx`. Build `X` from the number columns only (eleven columns), leaving out `default` and `collection_calls`, and build `y` from `default`.

In [ ]:
import pandas as pd

# Your turn: read loans.csv into loans
# Your turn: replace ages outside 18 to 90 with the median age
# Your turn: fill the blank monthly incomes with the median income
# Your turn: add the burden column
# Your turn: X holds the number columns of loans, without default and collection_calls
# Your turn: y is the default column


In [ ]:
_missing = [v for v in ('X', 'y',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert X.shape == (8000, 11), 'X should have 8000 rows and 11 columns: number columns only, with default and collection_calls left out'
assert X.isna().sum().sum() == 0, 'Fill the blank incomes with the median'
assert 'collection_calls' not in X.columns, 'collection_calls leaks the answer. Leave it out'
assert y.sum() == 1004, 'y should be the default column (1004 defaults)'
print('Task 1 correct')

## Task 2. Split, then a simple baseline
Split `X` and `y` 75 to 25 with `stratify=y` and `random_state=42`. Fit a logistic regression (scaled first, in a Pipeline) and store its test AUC in `auc_logit`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Your turn: split X and y into X_train, X_test, y_train, y_test using the settings in the task
# Your turn: logit is a pipeline that scales, then fits a logistic regression (max_iter=1000)
# Your turn: fit it on the training part
# Your turn: auc_logit_train and auc_logit are the AUC of its chances of default on the training part and on the test part


In [ ]:
_missing = [v for v in ('auc_logit', 'auc_logit_train',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 0.68 <= auc_logit <= 0.85, 'auc_logit should be the AUC on the test part, using the chance of default (predict_proba, column 1)'
assert abs(auc_logit_train - auc_logit) < 0.08, 'A simple model should have a small gap between train and test. Check which part you scored'
print('Task 2 correct')

## Task 3. A default forest: train against test
Fit a `RandomForestClassifier` with 200 trees and `random_state=42`. Store the train AUC in `auc_f_train` and the test AUC in `auc_f_test`. How big is the gap?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Your turn: make a forest as the task describes and call it forest
# Your turn: fit it on the training part
# Your turn: auc_f_train and auc_f_test, scored as in Task 2


In [ ]:
_missing = [v for v in ('auc_f_test', 'auc_f_train',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert auc_f_train >= 0.90, 'A default forest should score very high on its own training loans. Score X_train and y_train'
assert 0.68 <= auc_f_test <= 0.90, 'auc_f_test should be scored on the test part, not the training part'
assert auc_f_train > auc_f_test, 'The training score should be higher than the test score'
print('Task 3 correct')

## Task 4. Watch the gap open as depth grows
For `max_depth` in 2, 4, 6, 8, 12 and 20, fit a forest of 100 trees. Collect the train AUC in `train_scores` and the test AUC in `test_scores`, then plot both against depth. This is the Jinja curve again, on loans.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

depths = [2, 4, 6, 8, 12, 20]
train_scores = []
test_scores = []
for d in depths:
    # Your turn: fit a forest of 100 trees with max_depth=d
    # Your turn: add its train AUC to train_scores and its test AUC to test_scores
    pass
# Your turn: plot both lists against depths, with axis labels and a legend


In [ ]:
_missing = [v for v in ('test_scores', 'train_scores',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(train_scores) == 6 and len(test_scores) == 6, 'Add one score per depth to each list'
assert train_scores[-1] > train_scores[0], 'Train AUC should rise with depth. Score the training part for train_scores'
assert train_scores[-1] - test_scores[-1] > train_scores[0] - test_scores[0], 'The gap should be wider at depth 20 than at depth 2. Score the test part for test_scores'
print('Task 4 correct')

## Task 5. Tune with cross-validation
Use `GridSearchCV` with `max_depth` in 3, 6 and None and `min_samples_leaf` in 1, 10 and 30, scored by `roc_auc` over 5 folds. Store the fitted search in `search`, then the tuned forest's train and test AUC in `auc_t_train` and `auc_t_test`.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Your turn: grid is a dictionary with the two settings and their values from the task
# Your turn: search is a GridSearchCV around a forest of 200 trees (random_state=42), scored by roc_auc over 5 folds
# Your turn: fit the search on the training part
# Your turn: tuned is the best forest it found. Then fill auc_t_train and auc_t_test


In [ ]:
_missing = [v for v in ('auc_f_test', 'auc_f_train', 'auc_t_test', 'auc_t_train', 'search',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 0.65 <= search.best_score_ <= 0.90, 'search.best_score_ looks wrong. Fit the search on the training part only, with scoring=\'roc_auc\''
assert 0.68 <= auc_t_test <= 0.90, 'auc_t_test should be scored on the test part with the tuned forest'
assert (auc_t_train - auc_t_test) <= (auc_f_train - auc_f_test) + 0.01, 'The tuned forest should not overfit more than the default one. Check the grid values'
print('Task 5 correct')

## Task 6. A boosted model, and one table
Fit a `HistGradientBoostingClassifier` (learning_rate 0.05, max_iter 200, max_depth 3, random_state 42). Store its test AUC in `auc_b_test`. Then build a DataFrame `results` with one row per model (logistic regression, tuned forest, boosting) and columns `model` and `test_auc`.

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Your turn: make the boosted model with the settings in the task, call it boost, and fit it on the training part
# Your turn: auc_b_test is its test AUC
# Your turn: results is a DataFrame with columns model and test_auc, three rows


In [ ]:
_missing = [v for v in ('auc_b_test', 'results',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert 0.68 <= auc_b_test <= 0.90, 'auc_b_test should be the AUC on the test part, using the chance of default'
assert len(results) == 3, 'results needs one row per model: three rows'
assert list(results.columns) == ['model', 'test_auc'], 'results needs the columns model and test_auc'
print('Task 6 correct')

## Task 7. Which columns did the forest lean on?
Use `permutation_importance` on the tuned forest and the test part (10 repeats, `random_state=42`). Store the five largest in `top5`, a Series sorted from largest to smallest.

In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance

# Your turn: result is permutation_importance for tuned on the test part, scored by roc_auc, 10 repeats, random_state=42
# Your turn: imp is a Series of the mean importances with the column names as its index
# Your turn: top5 is the five largest, largest first


In [ ]:
_missing = [v for v in ('X', 'top5',) if v not in globals()]
assert not _missing, f'Not set yet: {_missing}. Finish the Your turn cell above, run it, then run this cell again.'
assert len(top5) == 5, 'top5 should keep five columns'
assert top5.iloc[0] >= top5.iloc[-1], 'Sort from largest to smallest'
assert set(top5.index) <= set(X.columns), 'The index of top5 should be column names. Pass index=X.columns when you build the Series'
print('Task 7 correct')

## Task 8. Decide, in your own words
Look at `results`, and at the train and test scores in Tasks 2, 3 and 5. Write two sentences below. First, did the complexity pay: how much better is the best model than the logistic regression, and is that gain worth losing the readable weights? Second, which model would you ship, and what would make you change your mind? Compare your numbers with the typical ones in the lesson. They should be close, not identical.